In [1]:
import numpy as np
import pandas as pd
import animations as anim

In [2]:
anim.generate()

generated in 33 min 31 s                                                                                          

In [3]:
np.random.seed(42)

In [4]:
train = pd.read_csv("data/cs-training.csv")
test = pd.read_csv("data/cs-test.csv")

In [5]:
train = train.drop(columns = ["Id"])
test = test.drop(columns = ["Id"])

In [6]:
for col in ["MonthlyIncome", "NumberOfDependents"]:
    train_median = train[col].median()
    train[col] = train[col].fillna(train_median)
    test[col] = test[col].fillna(train_median)

In [7]:
train["TotalDebt"] = train["DebtRatio"]*train["MonthlyIncome"]
test["TotalDebt"] = test["DebtRatio"]*test["MonthlyIncome"]

In [8]:
for col in ["MonthlyIncome", "DebtRatio"]:
    train[col] = np.log1p(train[col])
    test[col] = np.log1p(test[col])

In [9]:
for col in ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfTimes90DaysLate"]:
    train[col] = np.clip(train[col], 0, 20)
    test[col] = np.clip(test[col], 0, 20)

In [10]:
col = "RevolvingUtilizationOfUnsecuredLines"
train[col] = np.clip(train[col], 0, 1)
test[col] = np.clip(test[col], 0, 1)

In [11]:
X = train.drop("SeriousDlqin2yrs", axis = 1).values
y = train["SeriousDlqin2yrs"].values.reshape(-1, 1)

In [12]:
indices = np.random.permutation(len(X))
split = int(0.8*len(X))
train_idx = indices[:split]
val_idx = indices[split:]

In [13]:
X_train = X[train_idx]
y_train = y[train_idx]
X_val = X[val_idx]
y_val = y[val_idx]

In [14]:
X_mean = X_train.mean(axis = 0)
X_std = X_train.std(axis = 0)
X_train = (X_train - X_mean)/X_std
X_val = (X_val - X_mean)/X_std

In [15]:
classes, counts = np.unique(y, return_counts = True)
total = len(y)
class_weights = {c: total/(2*count) for c, count in zip(classes, counts)}
sample_weights = np.array([class_weights[yi[0]] for yi in y]).reshape(-1, 1)

In [16]:
layers = [X_train.shape[1], 8, 8, 8, 1]

In [17]:
weights = []
biases = []
for i in range(len(layers) - 1):
    w = np.random.randn(layers[i], layers[i + 1])*np.sqrt(2/layers[i])
    b = np.zeros((1, layers[i + 1]))
    weights.append(w)
    biases.append(b)

In [18]:
def sigmoid(x):
    return 1/(1 + np.exp(-x))

In [19]:
def relu(x):
    return np.maximum(0, x)

In [20]:
def heaviside(x):
    return (x > 0).astype(float)

In [21]:
def prob(x):
    a = x
    for i in range(len(weights) - 1):
        a = relu(np.dot(a, weights[i]) + biases[i])
    out = sigmoid(np.dot(a, weights[-1]) + biases[-1])
    return out

In [22]:
def roc_auc(y_true, y_score):
    desc_order = np.argsort(-y_score.flatten())
    y_true = y_true.flatten()[desc_order]
    y_score = y_score.flatten()[desc_order]
    tps = np.cumsum(y_true)
    fps = np.cumsum(1 - y_true)
    tpr = tps/tps[-1]
    fpr = fps/fps[-1]
    auc = np.trapezoid(tpr, fpr)
    return auc

In [23]:
epochs = 100
lr = 0.02
dr = 0.2
batch_size = 256
n_samples = X_train.shape[0]
loss_history = []
train_auc_history = []
val_auc_history = []

for epoch in range(epochs):
    epoch_loss = 0
    perm = np.random.permutation(n_samples)
    X_shuff = X_train[perm]
    y_shuff = y_train[perm]
    w_shuff = sample_weights[perm]
    
    for i in range(0, n_samples, batch_size):
        X_batch = X_shuff[i:i + batch_size]
        y_batch = y_shuff[i:i + batch_size]
        w_batch = w_shuff[i:i + batch_size]

        activations = [X_batch]
        curr_a = X_batch
        dropout_masks = []
        for j in range(len(weights) - 1):
            curr_a = relu(np.dot(curr_a, weights[j]) + biases[j])
            mask = (np.random.rand(*curr_a.shape) > dr).astype(float)
            curr_a *= mask
            curr_a /= (1 - dr)
            dropout_masks.append(mask)
            activations.append(curr_a)
        out = sigmoid(np.dot(curr_a, weights[-1]) + biases[-1])

        y1 = y_batch*np.log(out + 1e-8)
        y0 = (1 - y_batch)*np.log(1 - out + 1e-8)
        loss = -np.mean(w_batch*(y1 + y0))
        epoch_loss += loss*len(X_batch)

        delta = w_batch*(out - y_batch)/len(X_batch)
        for j in reversed(range(len(weights))):
            dW = np.dot(activations[j].T, delta)
            db = np.sum(delta, axis = 0, keepdims = True)
            if j > 0:
                delta = np.dot(delta, weights[j].T)
                delta *= heaviside(activations[j])
                delta *= dropout_masks[j - 1]
                delta /= (1 - dr)
            weights[j] -= lr*dW
            biases[j] -= lr*db

    epoch_loss /= n_samples
    loss_history.append(epoch_loss)

    train_pred = prob(X_train)
    train_auc = roc_auc(y_train, train_pred)
    train_auc_history.append(train_auc)

    val_pred = prob(X_val)
    val_auc = roc_auc(y_val, val_pred)
    val_auc_history.append(val_auc)